In [ ]:
#!pip uninstall tft_pytorch -y

In [ ]:
#!pip install -U --no-cache git+https://github.com/rsscml/tft_pytorch

In [ ]:
#!pip install -U openpyxl

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import gc
import time
import copy
import os
import random
import warnings
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 1000)
pd.set_option("display.max_rows", 1000)

import torch

from tft_pytorch import (
    OptimizedTFTDataset, 
    create_tft_dataloader, 
    create_uniform_embedding_dims,
    TemporalFusionTransformer,
    TFTTrainer,
    TFTInferenceWithTracking)

In [ ]:
df = pd.read_excel("../data/plant_mat_channel_agg_input.xlsx", dtype={"Plant": 'str', "Material": 'str', "Channel": 'str'})

In [ ]:
df['Month'] = df['YearMonth'].dt.month
df['Quarter'] = df['YearMonth'].dt.quarter
df['Customer Demand Qty'] = np.sqrt(df['Customer Demand Qty'])


In [ ]:
df.head()

In [ ]:
# BG Plant
df = df[df['Plant']=='3916']

df.shape

In [ ]:
# Describe features

features_config = {
    "entity_col": "key",
    "time_index_col": "YearMonth",
    "target_col": "Customer Demand Qty",

    # known in the future
    "temporal_known_numeric_col_list": ["Working_Days"],
    "temporal_known_categorical_col_list": ["Month", "Quarter","Minimum Lot Size"],

    # only historical
    "temporal_unknown_numeric_col_list": [],
    "temporal_unknown_categorical_col_list": [],

    # static per entity
    "static_numeric_col_list": [],
    "static_categorical_col_list": ["Material","Channel","Material_Group","Material_Hierarchy"]
}

In [ ]:

# context window length
historical_steps=24
forecast_steps=3
train_min_historical_steps=12
test_min_historical_steps=12
infer_min_historical_steps=0

# no. of samples
train_stride=1
test_stride=1

# scaler and encoder location
scaler_path='./artefacts/train_scalers.pkl'
scaling_strategy='entity_level'
scaling_method='mean'
encoders_path='./artefacts'

# parallel processing
train_n_jobs = 4
test_n_jobs = 4
infer_n_jobs = 2

# cutoff periods for train/val/infer
train_start = pd.to_datetime('2020-01-01', format="%Y-%m-%d")
train_cutoff = pd.to_datetime('2024-11-01', format="%Y-%m-%d")

test_start = train_cutoff - pd.DateOffset(months=historical_steps)
test_cutoff = pd.to_datetime('2025-05-01', format="%Y-%m-%d")

infer_cutoff = pd.to_datetime('2025-08-01', format="%Y-%m-%d")
infer_start = infer_cutoff - pd.DateOffset(months=historical_steps + forecast_steps)

print(f" train_start: {train_start}, train_cutoff: {train_cutoff}")
print(f" test_start: {test_start}, test_cutoff: {test_cutoff}")
print(f" infer_start: {infer_start}, infer_cutoff: {infer_cutoff}")

# create train & test datasets
df = df.sort_values(by=['key','YearMonth'], ascending=True)

train_df = df[df['YearMonth'] <= train_cutoff].copy()
test_df = df[(df['YearMonth'] > test_start) & (df['YearMonth'] <= test_cutoff)].copy()

# get last historical_steps + forecast_steps steps for inference

infer_df = df[df['YearMonth'] <= infer_cutoff].copy()
infer_df =  infer_df.groupby(['key'], group_keys=False).tail(historical_steps + forecast_steps)

print(train_df.shape, test_df.shape, infer_df.shape)


In [ ]:
# Create datasets

train_dataset = OptimizedTFTDataset(
                data_source=train_df,
                features_config=features_config, 
                historical_steps=historical_steps,
                prediction_steps=forecast_steps,
                stride=train_stride,
                enable_padding=True,
                padding_strategy = 'zero',
                categorical_padding_value = -1,
                min_historical_steps=train_min_historical_steps,
                allow_inference_only_entities=False,
                scaler_path=scaler_path,
                scaling_strategy=scaling_strategy,
                scaling_method=scaling_method,
                cold_start_scaler_cols=None,
                mean_scaler_epsilon=1.0,
                recency_alpha=1.0,
                n_jobs=train_n_jobs,
                max_series=None,
                mode='train',
                encoders_path=encoders_path,
                fit_encoders=None,
                preprocessing_fn=None)

test_dataset = OptimizedTFTDataset(
                data_source=test_df,
                features_config=features_config, 
                historical_steps=historical_steps,
                prediction_steps=forecast_steps,
                stride=test_stride,
                enable_padding=True,
                padding_strategy = 'zero',
                categorical_padding_value = -1,
                min_historical_steps=test_min_historical_steps,
                allow_inference_only_entities=False,
                scaler_path=scaler_path,
                scaling_strategy=scaling_strategy,
                scaling_method=scaling_method,
                cold_start_scaler_cols=None,
                mean_scaler_epsilon=1.0,
                recency_alpha=0,
                n_jobs=test_n_jobs,
                max_series=None,
                mode='test',
                encoders_path=encoders_path,
                fit_encoders=None,
                preprocessing_fn=None)

infer_dataset = OptimizedTFTDataset(
                data_source=infer_df,
                features_config=features_config, 
                historical_steps=historical_steps,
                prediction_steps=forecast_steps,
                stride=test_stride,
                enable_padding=True,
                padding_strategy = 'zero',
                categorical_padding_value = -1,
                min_historical_steps=infer_min_historical_steps,
                allow_inference_only_entities=True,
                scaler_path=scaler_path,
                scaling_strategy=scaling_strategy,
                scaling_method=scaling_method,
                cold_start_scaler_cols=['Plant','Material','Channel'], 
                mean_scaler_epsilon=1.0,
                recency_alpha=0,
                n_jobs=1,
                max_series=None,
                mode='test',
                encoders_path=encoders_path,
                fit_encoders=None,
                preprocessing_fn=None)

# create dataloaders
train_dataloader, train_adapter = create_tft_dataloader(train_dataset, 
                                                        batch_size=128, 
                                                        shuffle=True,
                                                        num_workers=0,
                                                        drop_last=False,
                                                        pin_memory=True)

test_dataloader, test_adapter = create_tft_dataloader(test_dataset, 
                                                      batch_size=128, 
                                                      shuffle=False,
                                                      num_workers=0,
                                                      drop_last=False,
                                                      pin_memory=True)

infer_dataloader, infer_adapter = create_tft_dataloader(infer_dataset,
                                                        batch_size=128,
                                                        shuffle=False,
                                                        num_workers=0,
                                                        drop_last=False,
                                                        pin_memory=True)

In [ ]:
# sample dataset outputs

#static_categorical: Optional[List[torch.Tensor]] = None,
#static_continuous: Optional[List[torch.Tensor]] = None,
#historical_categorical: Optional[List[torch.Tensor]] = None,
#historical_continuous: Optional[List[torch.Tensor]] = None,
#future_categorical: Optional[List[torch.Tensor]] = None,
#future_continuous: Optional[List[torch.Tensor]] = None,
#padding_mask: Optional[torch.Tensor] = None

for batch in infer_dataloader:
    model_inputs = infer_adapter.adapt_for_tft(batch)
    print("Model Input Keys: \n")
    for k, v in model_inputs.items():
        try:
            if isinstance(v, list):
                print(f"Key: {k}: type: List, num_elements: {len(v)}, shape: {v[0].shape}")
            else:
                print(f"Key: {k}: type: Tensor, shape: {v.shape}")
        except:
            print(f"Key: {k}")
    break

In [ ]:
# create embedding dims dict for various categorical variables
categorical_embedding_dims = create_uniform_embedding_dims(train_dataset, hidden_layer_size=160)

print("categorical embedding dims: \n", categorical_embedding_dims)

num_static_categorical=len(train_dataset.static_categorical_cols)
num_static_continuous=len(train_dataset.static_numeric_cols)
num_historical_categorical=len(train_dataset.temporal_unknown_categorical_cols) + len(train_dataset.temporal_known_categorical_cols)
num_historical_continuous=len(train_dataset.target_cols) + len(train_dataset.temporal_unknown_numeric_cols) + len(train_dataset.temporal_known_numeric_cols)
num_future_categorical=len(train_dataset.temporal_known_categorical_cols)
num_future_continuous=len(train_dataset.temporal_known_numeric_cols)

print("num_static_categorical: ", num_static_categorical)
print("num_static_continuous: ", num_static_continuous)
print("num_historical_categorical: ", num_historical_categorical)
print("num_historical_continuous: ", num_historical_continuous)
print("num_future_categorical: ", num_future_categorical)
print("num_future_continuous: ", num_future_continuous)


In [ ]:
# create Model

model = TemporalFusionTransformer(
        # Model architecture parameters
        hidden_layer_size = 160,
        num_attention_heads = 1,
        num_lstm_layers = 1,
        num_attention_layers = 4,
        dropout_rate = 0.1,
        
        # Input specification
        num_static_categorical = num_static_categorical,
        num_static_continuous = num_static_continuous,
        num_historical_categorical = num_historical_categorical,
        num_historical_continuous = num_historical_continuous,
        num_future_categorical = num_future_categorical,
        num_future_continuous = num_future_continuous,
    
        # embedding dims for categorical variables
        categorical_embedding_dims = categorical_embedding_dims,
        
        # Time dimensions
        historical_steps = historical_steps,
        prediction_steps = forecast_steps,
        
        # Output specification
        num_outputs = 1,
        device = 'cuda' #'cuda' if torch.cuda.is_available() else 'cpu'
    )

In [ ]:
# Train

trainer = TFTTrainer(model = model,
                     # data params
                     train_loader = train_dataloader,
                     val_loader = test_dataloader,
                     train_adapter = train_adapter,
                     val_adapter = test_adapter,
                     # loss params
                     loss_type = 'huber',
                     loss_params = {'delta': 1.0},
                     # optimizer params
                     optimizer_type = 'adam',
                     learning_rate = 0.0001,
                     # Scheduler configuration
                     scheduler_type = 'reduce_on_plateau',
                     scheduler_factor = 0.5,
                     scheduler_patience = 5,
                     # Training options
                     enable_gradient_clipping = True,
                     max_grad_norm = 1.0,
                     enable_train_sample_weighting = False,
                     enable_val_sample_weighting = False,
                     # Mixed Precision Training
                     enable_mixed_precision = False,
                     # Checkpointing
                     save_path = './models/plant_mat_ch',
                     save_every = 1)

In [ ]:
trainer.train(num_epochs=50, patience=10)

In [ ]:
inferencewithtracking = TFTInferenceWithTracking(model_path = 'models/plant_mat_ch/best_model.pt',
                                                 model = model,
                                                 adapter = infer_adapter,
                                                 device = 'cuda')

In [ ]:
results_df = inferencewithtracking.predict_with_metadata(infer_dataloader)

In [ ]:
results_df.head()

In [ ]:
# inverse power transform
results_df['pred_0'] = np.square(results_df['pred_0'])
results_df['actual_Customer Demand Qty'] = np.square(results_df['actual_Customer Demand Qty'])


In [ ]:
results_df.to_csv("./outputs/bg_huber_1_transform_plant_material_channel_aug.csv", index=False)